# 📊 Báo Cáo Khảo Sát & Phân Tích Thống Kê Dữ Liệu VN30 (2018–2025)
## Đề tài: Risk-Aware Reinforcement Learning for VN30 Portfolio Allocation (INT3011E)
---
### 1. Bối cảnh nghiên cứu & Mục tiêu chuẩn bị dữ liệu (O1)
Notebook này tổng hợp toàn bộ tập dữ liệu thô vừa thu thập qua `vnstock` cho 30 cổ phiếu đại diện rổ VN30 và 2 chỉ số tham chiếu (`VN30`, `VNINDEX`) từ năm 2018 đến 2025. Phân tích này là cơ sở thực nghiệm để thiết kế môi trường kế toán danh mục $M_1$ và 4 hàm thưởng nhận thức rủi ro $A_0 - A_3$.

In [ ]:
        import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Load summary metadata
summary_files = sorted(glob.glob('../data/raw/prices/*.csv'))
print(f'✅ Đã tải thành công {len(summary_files)} file CSV dữ liệu cổ phiếu VN30.')

events_df = pd.read_csv('../data/raw/corporate_actions/vn30_all_corporate_events.csv')
print(f'✅ Đã thu thập {len(events_df)} sự kiện doanh nghiệp (cổ tức tiền, cổ tức cổ phiếu, chia tách).')

: 

### 2. Quỹ đạo giá chuẩn hóa của 30 cổ phiếu VN30 & Các chu kỳ vĩ mô
Biểu đồ dưới đây thể hiện đường cong giá chuẩn hóa (mốc 100 tại đầu kỳ 2018). Chúng ta thấy rõ 4 giai đoạn thị trường tương ứng với 4 cửa sổ trượt (Rolling Windows):
- **Cửa sổ 1 (2018–2022):** Bao gồm cú sập COVID-19 (tháng 3/2020) và cú lao dốc kỷ lục năm 2022 (-42% do khủng hoảng trái phiếu và thắt chặt thanh khoản).
- **Cửa sổ 2–4 (2023–2025):** Giai đoạn phân hóa mạnh, phục hồi và đi ngang tích lũy.

In [ ]:
display(Image(filename='../outputs/figures/01_vn30_price_trajectories.png'))

### 3. Phân tích rủi ro sụt giảm (Underwater Drawdown Profile) của VN30-Index
Chỉ số VN30-Index từng trải qua các đợt sụt giảm từ đỉnh nghiêm trọng:
- **Tháng 03/2020:** Sụt giảm **-33.8%**.
- **Năm 2022:** Sụt giảm tới **-45.2%**.

> **Ý nghĩa đối với đề tài:** Các mô hình RL tối ưu hàm thưởng lợi suất truyền thống ($A_0$) thường tiếp tục nắm giữ cổ phiếu khi thị trường rơi tự do. Việc đưa rào cản phạt sụt giảm vượt ngưỡng $d^* = 15\%$ trong hàm thưởng $A_2$ và $A_3$ đóng vai trò cảnh báo tác tử rút bớt vốn về tiền mặt ($w_{\text{cash}} \to 1.0$).

In [ ]:
display(Image(filename='../outputs/figures/06_vn30_underwater_drawdown.png'))

### 4. Ma trận tương quan lợi suất chéo (Cross-Asset Correlation)
Ma trận thể hiện hệ số tương quan lợi suất ngày giữa 30 cổ phiếu. Các cổ phiếu cùng ngành (đặc biệt là Ngân hàng: `VCB`, `BID`, `CTG`, `MBB`, `TCB` và Bất động sản: `VIC`, `VHM`, `VRE`) có mức độ đồng pha cao ($
ho > 0.65$), tạo cơ hội cho thuật toán PPO học chính sách phân bổ luân chuyển dòng tiền thông minh.

In [ ]:
display(Image(filename='../outputs/figures/02_vn30_correlation_matrix.png'))

### 5. Phân phối lợi suất thực nghiệm & Rủi ro đuôi (Fat Tails & Kurtosis)
Thực nghiệm trên gần 60.000 mẫu lợi suất ngày của 30 cổ phiếu cho thấy:
- Phân phối lợi suất có đỉnh nhọn và **đuôi dày rõ rệt (Excess Kurtosis > 3.0)** so với phân phối chuẩn Gaussian.
- **Historical VaR 95%:** ~ -2.8% / ngày; **Historical CVaR 95%:** ~ -4.3% / ngày.
- Điều này chứng minh tính cấp thiết của hàm thưởng phạt lợi suất âm dưới ngưỡng $m$ ($A_1$) và kiểm soát tổn thất đuôi CVaR khi đánh giá.

In [ ]:
display(Image(filename='../outputs/figures/03_vn30_return_fat_tails.png'))

### 6. Hồ sơ thanh khoản (ADV) & Sự kiện doanh nghiệp (Corporate Actions)
- **Thanh khoản:** Các cổ phiếu dẫn đầu thanh khoản gồm `HPG`, `SSI`, `STB`, `MBB` đạt hàng chục triệu cổ phiếu/ngày, bảo đảm tính khả thi cho việc mô phỏng khớp lệnh thực tế $M_1$ (lô 100) và $M_2$ (giới hạn thanh khoản).
- **Sự kiện quyền:** 275 sự kiện chi trả cổ tức và thưởng cổ phiếu được lưu trữ đầy đủ để phục vụ module kế toán dòng tiền chính xác.

In [ ]:
display(Image(filename='../outputs/figures/04_vn30_liquidity_profile.png'))
display(Image(filename='../outputs/figures/05_vn30_corporate_actions.png'))